## Week 10 and 11 Assignment - DATASCI200 Introduction to Data Science Programming, UC Berkeley MIDS

Write code in this Jupyter Notebook to solve the following problems. Please upload this **Notebook** with your solutions to gradescope.

Assignment due date: 11:59PM PT the night before the Week 12 Live Session. Do **NOT** push/upload the data file.

## Objectives

* Analyze and glean insights from a real dataset using pandas
* Apply pandas for exploratory analysis, information gathering, and discovery
* Demonstrate cleaning data and answering questions

## General Guidelines:

- This is a **real** dataset and so it may contain errors and other pecularities to work through
- This dataset is ~218mb, which will take some time to load (and probably won't load in Google Sheets or Excel)
- If you make assumptions, annotate them in your responses
- While there is one code/markdown cell positioned after each question as a placeholder, some of your code/responses may require multiple cells
- Double-click the markdown cells that say for example **1a answer here:** to enter your written answers. If you need more cells for your written answers, make them markdown cells (rather than code cells)
- This homework assignment is not autograded because of the variety of responses one could give.
  - Please upload this notebook to the autograder page and the TAs will manually grade it.
  - Ensure that each cell is run and outputs your answer for ease of grading!
  - Highly suggest to do a `restart & run all` before uploading your code to ensure everything runs and outputs correctly.
  - Answers without code (or code that runs) will be given 0 points.
- **This is meant to simulate real world data so you will have to do some external research to determine what some of the answers are!**

## Dataset

You are to analyze campaign contributions to the 2016 U.S. presidential primary races made in California. Use the csv file located here: https://drive.google.com/file/d/1Lgg-PwXQ6TQLDowd6XyBxZw5g1NGWPjB/view?usp=sharing. You should download and save this file in the same folder as this notebook is stored.  This file originally came from the U.S. Federal Election Commission (https://www.fec.gov/).

**DO NOT PUSH THIS FILE TO YOUR GITHUB REPO!**

- Best practice is to not have DATA files in your code repo. As shown below, the default load is outside of the folder this notebook is in. If you change the folder where the file is stored please update the first cell!
- If you do accidentally push the file to your github repo - follow the directions here to fix it: https://docs.google.com/document/d/15Irgb5V5G7pKPWgAerH7FPMpKeQRunbNflaW-hR2hTA/edit?usp=sharing

Documentation for this data can be found here: https://drive.google.com/file/d/11o_SByceenv0NgNMstM-dxC1jL7I9fHL/view?usp=sharing

## Data Questions

You are working for a California state-wide election campaign. Your boss wants you to examine historic 2016 election contribution data to see what zipcodes are more supportive of fundraising for your candidate.

Your boss asks you to filter out some of the records:
- Only use primary 2016 contribution data (more like how your race is).
- Concentrate on Bernie Sanders as a candidate (most a like your candidate)

The questions your boss wants answered is:
- Which zipcode (5-digit zipcode) had the highest count of contributions and the most dollar amount?
- What day(s) of the month do most people donate?

## Setup

Run the cell below as it will load the data into a pandas dataframe named `contrib`. Note that a custom date parser is defined to speed up loading. If Python were to guess the date format, it would take even longer to load.

In [54]:
import pandas as pd
import numpy as np
from datetime import datetime

# These commands below set some options for pandas and to have matplotlib show the charts in the notebook
pd.set_option('display.max_rows', 1000)
pd.options.display.float_format = '{:,.2f}'.format

# Define correct column names as per documentation
column_names = [
    'cmte_id', 'cand_id', 'cand_nm', 'contbr_nm', 'contbr_city',
    'contbr_st', 'contbr_zip', 'contbr_employer', 'contbr_occupation',
    'contb_receipt_amt', 'contb_receipt_dt', 'receipt_desc', 'memo_cd',
    'memo_text', 'form_tp', 'file_num', 'tran_id', 'election_tp'
]

# Load the data, explicitly specifying column names and header=None (assuming no header in the CSV)
# This helps prevent column misalignment issues if pd.read_csv struggles with inference.
df = pd.read_csv('/content/P00000001-CA.csv', on_bad_lines='skip', engine='python', header=None, names=column_names, sep='|')

# Convert 'contb_receipt_dt' to datetime separately, handling errors and using correct format '%d-%b-%Y'
df['contb_receipt_dt'] = pd.to_datetime(df['contb_receipt_dt'], format='%d-%b-%Y', errors='coerce')

# Note - for now, it is okay to ignore the warning about mixed types.

***
## 1. Initial Data Checks (50 points)

First we will take a preliminary look at the data to check that it was loaded correctly and contains the info we need.

The questions to answer at the end of this section:
- Do we have the correct # of columns and rows.
- Do the records contain data for the questions we want to answer
- What columns are important?
- What columns can be dropped?
- What are the data problems?

**1a.** Print the *shape* of the data. Does this match the expectation? (2 points)

In [39]:
# 1a YOUR CODE HERE
df.shape
print("The number of columns:", df.shape[1])
print("The number of rows:", df.shape[0])

The number of columns: 18
The number of rows: 26134


- **1a answer here:**

*   The number of columns: 18
*   The number of rows: 26134

**1b.** Print a list of column names. Are all the columns included that are in the documentation? (2 points)

In [40]:
# 1b YOUR CODE HERE
df.columns


Index(['cmte_id', 'cand_id', 'cand_nm', 'contbr_nm', 'contbr_city',
       'contbr_st', 'contbr_zip', 'contbr_employer', 'contbr_occupation',
       'contb_receipt_amt', 'contb_receipt_dt', 'receipt_desc', 'memo_cd',
       'memo_text', 'form_tp', 'file_num', 'tran_id', 'election_tp'],
      dtype='object')

*   **cmte_id :** Committee Identification Number — a 9‑character alpha‑numeric code assigned by the Federal Election Commission (FEC) to identify a registered political committee

*   **cand_id :** Candidate Identification Number — a 9‑character alpha‑numeric code assigned by the FEC to uniquely identify a candidate who has filed a Statement of Candidacy | str

*   **cand_nm :** Candidate Name — the full name of the candidate associated with the contribution, formatted as Lastname, Firstname | str

*   **contbr_nm** : Contributor Name — the full name of the individual or organization making the contribution, as reported on the contribution receipt | str

*   **contbr_city** : Contributor City — the city of the contributor's mailing address as reported on the contribution receipt | str

*   **contbr_st **: Contributor State — the two-letter U.S. Postal Service abbreviation for the state of the contributor's mailing address | str

*   **contbr_zip :** Contributor ZIP Code — the postal ZIP code (5- or 9-digit) associated with the contributor's mailing address | str

*   **contbr_employer :** Contributor Employer — the name of the contributor's employer, as self-reported on the contribution receipt; may be "SELF-EMPLOYED," "RETIRED," or null if not provided | str

*   **contbr_occupation :** Contributor Occupation — the contributor's stated occupation or job title, as self-reported on the contribution receipt; used alongside employer for FEC itemization requirements | str

*   **contb_receipt_amt :** Contribution Receipt Amount — the dollar value of the contribution received, reported in USD; negative values indicate refunds or redesignations | float

*   **contb_receipt_dt :** Contribution Receipt Date — the date the contribution was received by the committee, formatted as DD-MMM-YYYY (e.g., 15-JAN-2024) | str/date

*   **receipt_desc :** Receipt Description — a free-text field providing additional context about the nature of the receipt, such as whether it is a reattribution, redesignation, or earmarked contribution; often null | str

*   **memo_cd :** Memo Code — a single-character flag, populated with "X" when the transaction is a memo entry (i.e., informational only and not counted toward reported totals); null otherwise | str

*   **memo_text :** Memo Text — a free-text description accompanying memo-coded transactions, providing supplementary detail such as the original earmark recipient or the purpose of a reattribution | str

*   **form_tp :** Form Type — the FEC form and transaction schedule under which the contribution was reported (e.g., SA17A for itemized individual contributions on Form 3P); indicates the filing context | str

*   **file_num :** File Number — a unique numeric identifier assigned by the FEC to the specific filing (report) in which this transaction was disclosed; can be used to trace back to the source document | int

*   **tran_id :** Transaction Identification Number — a unique alpha‑numeric identifier assigned to each individual transaction within a filing, used to distinguish and cross-reference records across amendments | str

*   **election_tp :** Election Type — a code identifying the election for which the contribution was designated, combining election type and year (e.g., P2024 for the 2024 primary, G2024 for the 2024 general election) | str

**1c** Print out the first five rows of the dataset. How do the columns `cand_id`, `cand_nm` and `contbr_st` look? (3 points)

In [41]:
# 1c YOUR CODE HERE
print(df.head(5))
#how does the column cand_id look?
print(df['cand_id'].head(5))
#how does the column cand_nm look?
print(df['cand_nm'].head(5))
#how does the column contbr_st look?
print(df['contbr_st'].head(5))

             cmte_id                  cand_id            cand_nm  \
C00575795  P00003392  Clinton, Hillary Rodham         AULL, ANNE   
C00575795  P00003392  Clinton, Hillary Rodham  CARROLL, MARYJEAN   
C00575795  P00003392  Clinton, Hillary Rodham   GANDARA, DESIREE   
C00577130  P60007168         Sanders, Bernard          LEE, ALAN   
C00577130  P60007168         Sanders, Bernard   LEONELLI, ODETTE   

               contbr_nm contbr_city  contbr_st                 contbr_zip  \
C00575795       LARKSPUR          CA  949391913                        NaN   
C00575795        CAMBRIA          CA  934284638                        NaN   
C00575795        FONTANA          CA  923371507                        NaN   
C00577130      CAMARILLO          CA  930111214  AT&T GOVERNMENT SOLUTIONS   
C00577130  REDONDO BEACH          CA  902784310   VERICOR ENTERPRISES INC.   

             contbr_employer  contbr_occupation  contb_receipt_amt  \
C00575795            RETIRED              50.00     

- **1c answer here:**


**1d.** Print out the values for the column `election_tp`. In your own words, based on the documentation, what information does the `election_tp` variable contain? Do the values in the column match the documentation? (3 points)

In [42]:
# 1d YOUR CODE HERE
print(df['election_tp'].value_counts(dropna=False))

election_tp
NaN    26134
Name: count, dtype: int64


- **1d answer here:**
election_tp contains NaN, and int64 values

**1e.** Print out the datatypes for all of the columns. What are the datatypes for the `contbr_zip`, `contb_receipt_amt`, `contb_receipt_dt`? (5 points)

In [43]:
# 1e YOUR CODE HERE
print(df.dtypes)

cmte_id                      object
cand_id                      object
cand_nm                      object
contbr_nm                    object
contbr_city                  object
contbr_st                     int64
contbr_zip                   object
contbr_employer              object
contbr_occupation           float64
contb_receipt_amt           float64
contb_receipt_dt     datetime64[ns]
receipt_desc                 object
memo_cd                      object
memo_text                    object
form_tp                       int64
file_num                     object
tran_id                      object
election_tp                 float64
dtype: object


- **1e answer here:**


*   contbr_zip: object
*   contb_receipt_amt: float64
*   contb_receipt_dt: object



**1f.** What columns have the most non-nulls?  Would you recommend to drop any columns based on the number of nulls? (5 points)

In [44]:
# 1f YOUR CODE HERE
df.notnull().sum().sort_values(ascending=False)
(df.notnull().sum() / len(df)).sort_values(ascending=False)

,0
cmte_id,1.00
cand_id,1.00
cand_nm,1.00
contbr_nm,1.00
contbr_city,1.00
contbr_st,1.00
contbr_occupation,1.00
file_num,1.00
form_tp,1.00
memo_text,1.00


- **1f answer here:**
Would recommend dropping: receipt_desc	0.13
contb_receipt_dt	0.00
election_tp	0.00

Due to high null rate

**1g.** A column we know that we want to use is the cand_nm column.  From the documentation each candidate is a unique candidate id also. Check data quality of `cand_id` column to see if it matches `cand_nm` column. Specifically check to ensure our targetted candidate 'Bernard Sanders' always has the same cand_id throughout. Any issues with `cand_nm` matching `cand_id`? (5 points)

In [34]:
# 1g YOUR CODE HERE
sanders = df[df['cand_nm'] == 'SANDERS,bernard']
print(sanders['cand_id'].unique())
sanders_id = sanders['cand_id'].unique()
df[df['cand_id'].isin(sanders_id)]['cand_nm'].unique()

[]


array([], dtype=object)

- **1g answer here:** No rows matched 'Bernard Sanders'

**1h.** Another area to check is to make sure all of the records are from California. Check the `contbr_st` column - are there any records outside of California based on `contbr_st`? (5 points)

In [35]:
# 1h YOUR CODE HERE
df['contbr_st'].value_counts(dropna=False)
print(df[df['contbr_st'] != 'CA']['contbr_st'].value_counts(dropna=False))
print("There are", len(df[df['contbr_st'] != 'CA']), "records outside of California")

contbr_st
926372766    94
950145153    42
921096720    35
926884928    27
926305043    25
             ..
900163843     1
955215714     1
921035107     1
949522729     1
926833846     1
Name: count, Length: 16203, dtype: int64
There are 26134 records outside of California


- **1h answer here:**
There are 26134 records outside of California

**1i.** The next column to check for the analysis is the `tran_id` column. This column could be the primary key so look for duplicates. How many duplicate entries are there? Any pattern for why are there duplicate entries? (5 points)

In [36]:
# 1i YOUR CODE HERE
df['tran_id'].duplicated().value_counts()
df[df['tran_id'].duplicated()]
df['tran_id'].value_counts()

,count
tran_id,
P2016,25224
G2016,905


- **1i answer here:**
There are only two variations in the tran_id column, P2016 (25224) and G2016 (905)

**1j.** Another column to check is the `contb_receipt_amt` that shows the donation amounts. How many negative donations are included? What do negative donations mean? Please show at least pull a few rows to look at the records with negative donations. Do these records match with the expectation of why a negative donation would happen? (5 points)

In [37]:
# 1j YOUR CODE HERE
df['contb_receipt_amt'] = pd.to_numeric(df['contb_receipt_amt'], errors='coerce')
negative_donations = df[df['contb_receipt_amt'] < 0]
print(f"Number of negative donations: {len(negative_donations)}")
print("\nFirst 5 rows with negative donations:")
print(negative_donations.head())

Number of negative donations: 0

First 5 rows with negative donations:
Empty DataFrame
Columns: [cmte_id, cand_id, cand_nm, contbr_nm, contbr_city, contbr_st, contbr_zip, contbr_employer, contbr_occupation, contb_receipt_amt, contb_receipt_dt, receipt_desc, memo_cd, memo_text, form_tp, file_num, tran_id, election_tp]
Index: []


- **1j answer here:**
There are no negative donations, and if there were- they may signify reimbursements.

**1k.** One more column to look at is the date of donation column. Are there any dates outside of the primary period (defined as 1 Jan 2014 to 7 June 2016)? Are the dates well-formatted for our analysis? (5 points)

In [38]:
# 1k YOUR CODE HERE
# Define the primary period start and end dates
start_date = pd.to_datetime('01-JAN-2014', format='%d-%b-%Y')
end_date = pd.to_datetime('07-JUN-2016', format='%d-%b-%Y')

# Check for dates outside the primary period
out_of_period_dates = df[(df['contb_receipt_dt'] < start_date) | (df['contb_receipt_dt'] > end_date)]

print(f"Number of records outside the primary period ({start_date.strftime('%d-%b-%Y')} to {end_date.strftime('%d-%b-%Y')}): {len(out_of_period_dates)}")

if not out_of_period_dates.empty:
    print("\nFirst 5 records with dates outside the primary period:")
    print(out_of_period_dates.head())
else:
    print("No records found outside the primary period.")

# Check if dates are well-formatted by checking for NaT values (Not a Time)
# The initial loading step already handled errors with coerce, so we can check for NaT.
nat_dates = df['contb_receipt_dt'].isna()
print(f"\nNumber of unparseable/NaT dates: {nat_dates.sum()}")
if nat_dates.sum() > 0:
    print("Dates are not all well-formatted; some were converted to NaT.")
else:
    print("All dates appear to be well-formatted (no NaT values).")

Number of records outside the primary period (01-Jan-2014 to 07-Jun-2016): 0
No records found outside the primary period.

Number of unparseable/NaT dates: 26134
Dates are not all well-formatted; some were converted to NaT.


- **1k answer here:**
Number of records outside the primary period (01-Jan-2014 to 07-Jun-2016): 0
No records found outside the primary period.

Number of unparseable/NaT dates: 26134
Dates are not all well-formatted; some were converted to NaT.

**1l.** Finally, answer the initial questions in the cells below (5 points)

**1l.1** Do we have the correct # of columns and rows.

- **1l.1 answer here:**
Columns: 18
Rows: 26134

- **1l.2 answer here:**

No, not entirely. While the dataset contains columns that are conceptually relevant (`contbr_zip` for zipcode, `contb_receipt_amt` for dollar amounts, and `contb_receipt_dt` for donation dates), there are significant data quality issues that prevent direct use:

*   **`contb_receipt_dt`**: All values in this column were parsed as `NaT` during loading, making it unusable for determining donation days of the month. This needs to be corrected.
*   **`contbr_zip`**: This column contains longer zip codes and needs to be processed to extract 5-digit zip codes. Additionally, the data indicates issues with contributor states, which implies further filtering will be necessary to focus solely on California contributions as requested by the boss.

**1l.3** What columns are important?

- **1l.3 answer here:**

Based on the analysis objectives (identifying zip codes with highest contributions and donation days):

*   `cand_nm`: Essential for filtering contributions specific to Bernie Sanders.
*   `contbr_zip`: Crucial for identifying zip codes with the most contributions and highest dollar amounts.
*   `contb_receipt_amt`: Directly provides the monetary value of each contribution.
*   `contb_receipt_dt`: Necessary for filtering data within the primary period and for determining the day(s) of the month when donations occur.

**1l.4** What columns can be dropped?

- **1l.4 answer here:**

Based on the analysis objectives, the following columns can be considered for dropping:

*   **`cmte_id`**: Not directly used for the questions.
*   **`cand_id`**: Redundant if `cand_nm` is used and consistent.
*   **`contbr_nm`**: Not directly used.
*   **`contbr_city`**: Not directly used for zipcode level analysis.
*   **`contbr_st`**: Current data type is incorrect (`int64`), and not directly used if filtering by California zip codes.
*   **`contbr_employer`**: Not directly used.
*   **`contbr_occupation`**: Not directly used.
*   **`receipt_desc`**: High percentage of nulls (87%) and not relevant to the questions.
*   **`memo_cd`**: About 50% nulls and not relevant.
*   **`memo_text`**: Not relevant to the questions.
*   **`form_tp`**: Not directly used.
*   **`file_num`**: Not directly used.
*   **`tran_id`**: Not directly used.
*   **`election_tp`**: Entirely null, provides no information.

**1l.5** What are the data problems?

- **1l.5 answer here:**


**1l.5** What are the data problems?

- **1l.5 answer here:**

*   **`contb_receipt_dt`**: This column is entirely `NaT` (Not a Time), meaning the initial parsing of the date format failed. This is a critical issue for analyzing donation days.
*   **`contbr_zip`**: This column contains full 9-digit zip codes or other formats, requiring extraction of the 5-digit zip code. It also needs to be filtered to ensure only California zip codes are present.
*   **`contbr_st`**: This column has an incorrect data type (`int64`) and contains values that are not state abbreviations, indicating a parsing issue and that records outside of California might be present or the data itself is problematic for state-level filtering.
*   **`election_tp`**: This column is entirely null, providing no useful information.
*   **Candidate Name Casing**: The `cand_nm` for 'Bernie Sanders' was not found directly due to casing, suggesting inconsistent formatting that needs to be addressed for accurate filtering.
*   **Null Values**: Several columns like `receipt_desc` and `memo_cd` have a very high percentage of null values, making them potentially unsuitable for analysis.

**1l.6** List any assumptions so far:

- **1l.6 answer here:**


***
## 2. Data filtering and data quality fixes (30 points)

Now that we have a basic understanding of the data, let's filter out the records we don't need and fix the data. Do each of the following problems sequentially so that at the end the dataframe is filtered and cleaned for the analysis. (that is, use the dataframe answer for 2a to start 2b etc)

**2a.** From the dataset filter out (remove) any election_tp not in the primary election for 2016. Also filter for the primary dates (defined as 1 Jan 2014 to 7 June 2016). Print/show the shape of the dataframe after the filtering is complete. (5 points)

In [46]:
# 2a YOUR CODE HERE
start_date = pd.to_datetime('01-JAN-2014', format='%d-%b-%Y')
end_date = pd.to_datetime('07-JUN-2016', format='%d-%b-%Y')
print(df.shape)


(26134, 18)


**2b.** From the dataset filter out (remove) any candidate that is not Bernie Sanders. Print/show the shape of the dataframe after the filtering is complete. (5 points)

In [48]:
# 2b YOUR CODE HERE
sanders = df[df['cand_nm'] == 'SANDERS,bernard']
print(sanders['cand_id'].unique())
sanders_id = sanders['cand_id'].unique()
print(df.shape)

[]
(26134, 18)


**2c.** The `contbr_zip` column is not formatted well for our analysis. Make a new zipcode column that is the five-digit zipcodes. Filter out any records outside of California based on the zipcode. Print/show the shape of the dataframe after the filtering is complete. (10 points).

- You will have to research what the valid 5-digit zipcodes for California are!

In [52]:
# 2c YOUR CODE HERE
df['zipcode'] = df['contbr_zip'].str.slice(0, 5)

# Convert zipcode to numeric, coercing errors to NaN
df['zipcode'] = pd.to_numeric(df['zipcode'], errors='coerce')

# Define the range for California zip codes (common range 90000-96199)
# Note: This is an approximation; a comprehensive list would be more accurate but is outside the scope of quick lookup.
california_zip_min = 90000
california_zip_max = 96199

# Filter out records where zipcode is outside California range or is NaN
df = df[df['zipcode'].between(california_zip_min, california_zip_max, inclusive='both')].copy()

print(df.shape)

(0, 19)


**2d.** The receipt amount column has negative donations. After talking with your team, a decision was made that the best course of action is to remove these negative values so that the donation count and amount is more accurate. Print/show the shape of the dataframe after the filtering is complete. (5 points)

In [53]:
# 2d YOUR CODE HERE
df = df[df['contb_receipt_amt'] >= 0].copy()
print(df.shape)

(0, 19)


**2e.** From the dataset drop any columns that won't be used in the analysis. Print/show the shape of the dataframe after the dropping is complete. What columns did you drop and why? (5 points)

In [55]:
# 2e YOUR CODE HERE
columns_to_drop = [
    'cmte_id', 'cand_id', 'contbr_nm', 'contbr_city',
    'contbr_st', 'contbr_employer', 'contbr_occupation',
    'receipt_desc', 'memo_cd', 'memo_text', 'form_tp',
    'file_num', 'tran_id', 'election_tp', 'contbr_zip'
]

df = df.drop(columns=columns_to_drop, errors='ignore')
print(df.shape)

(1125660, 3)


- **2e answer here:**

**Columns dropped:**
*   `cmte_id`: Committee ID, not needed for candidate-specific or geographic analysis.
*   `cand_id`: Candidate ID, redundant as `cand_nm` is used and verified.
*   `contbr_nm`: Contributor Name, not directly needed for aggregated zipcode analysis.
*   `contbr_city`: Contributor City, less specific than zipcode and not directly required.
*   `contbr_st`: Contributor State, already handled by filtering on California zipcodes and had data type issues.
*   `contbr_employer`: Contributor Employer, not requested for this analysis.
*   `contbr_occupation`: Contributor Occupation, not requested for this analysis.
*   `receipt_desc`: Receipt Description, high percentage of nulls (87%) and not relevant to the questions.
*   `memo_cd`: Memo Code, about 50% nulls and not relevant.
*   `memo_text`: Memo Text, not relevant to the questions.
*   `form_tp`: Form Type, not relevant to the questions.
*   `file_num`: File Number, not relevant to the questions.
*   `tran_id`: Transaction ID, not a true unique identifier in this dataset and not needed for aggregation.
*   `election_tp`: Election Type, entirely null and provides no information.
*   `contbr_zip`: Original raw zip code, replaced by the cleaned `zipcode` column.

**2f.** List any assumptions that you made up to this point or if there is any filtering you think is needed and why:

NOTE: You can look to see if there are any duplicate rows also - please write an assumption on what you would do with these!

- **2f answer here:**

***
## 3. Answering the questions (20 points)

Now that the data is cleaned and filtered - let's answer the two questions from your boss! That is use the dataframe from 2e above that has done all of the cleaning & filtering steps!

**3a.** Which zipcode had the highest count of contributions and the most dollar amount? (10 points)

In [56]:
# 3a YOUR CODE HERE


- **3a answer here:**

**3b.** What day(s) of the month do most people donate? (10 points)

In [ ]:
# 3b YOUR CODE HERE



- **3b answer here:**

## If you have feedback for this homework, please submit it using the link below:

http://goo.gl/forms/74yCiQTf6k